# EDA — `br_inep_ideb` por escola: dá pra virar a variável-alvo?

Contexto: confirmei que a `escola_completo` (Censo Escolar, 455 colunas) não tem nenhum indicador de resultado educacional nativo, e que o JOIN dela com `alunos` por `id_escola` é inviável (dois sistemas de código diferentes, sem crosswalk). Pesquisando fontes externas de desempenho por escola, encontrei o dataset `br_inep_ideb` e providenciei a ingestão dele no S3. O catálogo de colunas ficou assim:

| Coluna | Descrição |
|---|---|
| `ano` | Ano |
| `sigla_uf` | Sigla da UF |
| `id_municipio` | ID Município (IBGE, 7 dígitos) |
| `id_escola` | ID Escola (INEP) |
| `rede` | Rede escolar |
| `ensino` | Tipo de ensino |
| `anos_escolares` | Anos escolares (etapa) |
| `taxa_aprovacao` | Taxa de aprovação |
| `indicador_rendimento` | Indicador de rendimento |
| `nota_saeb_matematica` | Nota SAEB - Matemática |
| `nota_saeb_lingua_portuguesa` | Nota SAEB - Língua Portuguesa |
| `nota_saeb_media_padronizada` | Nota SAEB - Média padronizada |
| `ideb` | IDEB (média padronizada × indicador de rendimento) |
| `projecao` | Projeção |

**Este notebook responde exatamente as duas perguntas que preciso pra decidir se esse caminho é viável:**

1. Dá pra montar uma variável-alvo a partir daqui — preenchimento e distribuição de `taxa_aprovacao`/`ideb`, e cobertura da etapa "anos iniciais" (a mais próxima do recorte de alfabetização que uso desde o Dia 1 — atenção: o IDEB oficial é calculado no fim dos anos iniciais via SAEB do 5º ano, não é a mesma prova de alfabetização do 1º/2º ano que já uso, então isso é um proxy, não o mesmo indicador).
2. O `id_escola` daqui bate de verdade com o `id_escola` da `escola_completo` — checagem real (amostra, prefixos, interseção), a mesma disciplina que usei quando descobri que `alunos` e `escola` não batiam.

Não decido nada de arquitetura neste notebook — só levanto os números reais.

In [ ]:
import sys
sys.path.append("../..")

import boto3
import pandas as pd
import numpy as np
from pathlib import Path

from src.preprocessing.load_data import BUCKET, _ler_parquet_do_prefixo

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 100)

## 1. Carga do `br_inep_ideb/escola` (com cache local)

Uso só o path de escola — os paths de `brasil/`, `municipio/`, `regiao/` e `uf/` do mesmo dataset eu não preciso agora, já que meu grão de interesse é escola.

In [ ]:
PREFIXO_BRONZE_IDEB_ESCOLA = "bronze/br_inep_ideb/escola/"
CACHE_IDEB_ESCOLA = Path("../..") / "data" / "processed" / "ideb_escola.parquet"

if CACHE_IDEB_ESCOLA.exists():
    print(f"Lendo do cache local: {CACHE_IDEB_ESCOLA}")
    ideb_escola = pd.read_parquet(CACHE_IDEB_ESCOLA)
else:
    print(f"Cache não encontrado, lendo do S3: s3://{BUCKET}/{PREFIXO_BRONZE_IDEB_ESCOLA}")
    ideb_escola = _ler_parquet_do_prefixo(BUCKET, PREFIXO_BRONZE_IDEB_ESCOLA)
    CACHE_IDEB_ESCOLA.parent.mkdir(parents=True, exist_ok=True)
    ideb_escola.to_parquet(CACHE_IDEB_ESCOLA, index=False)
    print(f"Cache salvo em: {CACHE_IDEB_ESCOLA}")

print(f"\nideb_escola: {ideb_escola.shape[0]:,} linhas, {ideb_escola.shape[1]} colunas")
print(f"Colunas: {list(ideb_escola.columns)}")
ideb_escola.head(10)

## 2. Qual é o grão real dessa tabela?

Não vou assumir que é "1 linha por escola" só porque parece — confiro se existem múltiplas linhas por `(id_escola, ano)`, e por quê (provavelmente `rede`/`ensino`/`anos_escolares` quebram a granularidade, já que uma escola pode oferecer mais de uma etapa de ensino).

In [ ]:
print("Linhas totais:", f"{len(ideb_escola):,}")
print("id_escola distintos:", f"{ideb_escola['id_escola'].nunique():,}")
print("(id_escola, ano) distintos:", f"{ideb_escola.groupby(['id_escola', 'ano']).ngroups:,}")
print("(id_escola, ano, rede, ensino, anos_escolares) distintos:",
      f"{ideb_escola.groupby(['id_escola', 'ano', 'rede', 'ensino', 'anos_escolares']).ngroups:,}")

duplicado_por_ano = ideb_escola.groupby(["id_escola", "ano"]).size()
print(f"\nMáximo de linhas para o mesmo (id_escola, ano): {duplicado_por_ano.max()}")
print(f"Escolas/anos com mais de 1 linha: {(duplicado_por_ano > 1).sum():,} de {len(duplicado_por_ano):,}")

print("\nValores distintos de 'ano':", sorted(ideb_escola["ano"].unique()))
print("\nValores distintos de 'rede':")
print(ideb_escola["rede"].value_counts(dropna=False))
print("\nValores distintos de 'ensino':")
print(ideb_escola["ensino"].value_counts(dropna=False))
print("\nValores distintos de 'anos_escolares':")
print(ideb_escola["anos_escolares"].value_counts(dropna=False))

## 3. Preenchimento e distribuição dos candidatos a variável-alvo

`taxa_aprovacao`, `indicador_rendimento`, as notas do SAEB e o `ideb` em si — preciso saber quanto de cada um está realmente preenchido antes de considerar qualquer um deles usável.

In [ ]:
colunas_resultado = [
    "taxa_aprovacao",
    "indicador_rendimento",
    "nota_saeb_matematica",
    "nota_saeb_lingua_portuguesa",
    "nota_saeb_media_padronizada",
    "ideb",
    "projecao",
]

resumo_resultado = pd.DataFrame({
    "coluna": colunas_resultado,
    "qtd_preenchido": [ideb_escola[c].notna().sum() for c in colunas_resultado],
    "qtd_nulo": [ideb_escola[c].isna().sum() for c in colunas_resultado],
})
resumo_resultado["pct_preenchido"] = (resumo_resultado["qtd_preenchido"] / len(ideb_escola) * 100).round(2)
print(resumo_resultado)

print("\nDistribuição de 'taxa_aprovacao':")
print(ideb_escola["taxa_aprovacao"].describe())

print("\nDistribuição de 'ideb':")
print(ideb_escola["ideb"].describe())

## 4. Foco nos anos iniciais — a etapa mais próxima da alfabetização

O IDEB oficial dos anos iniciais é calculado com o SAEB do 5º ano — não é a mesma prova de alfabetização (1º/2º ano) que já uso desde o Dia 1, mas é a etapa mais próxima disponível aqui. Filtro pra ver especificamente essa fatia: quantas linhas sobram, e como fica o preenchimento e a distribuição dentro só desse recorte (o valor exato de `anos_escolares` pra filtrar eu vejo no `value_counts` da Seção 2, acima — ajusto aqui se o texto não for exatamente "Anos Iniciais").

In [ ]:
valor_anos_iniciais = [
    v for v in ideb_escola["anos_escolares"].dropna().unique()
    if "inicia" in str(v).lower()
]
print("Valor(es) de 'anos_escolares' identificado(s) como anos iniciais:", valor_anos_iniciais)

if valor_anos_iniciais:
    ideb_anos_iniciais = ideb_escola[ideb_escola["anos_escolares"].isin(valor_anos_iniciais)]
    print(f"\nLinhas nos anos iniciais: {len(ideb_anos_iniciais):,} de {len(ideb_escola):,}")
    print(f"Escolas distintas nos anos iniciais: {ideb_anos_iniciais['id_escola'].nunique():,}")
    print(f"Anos disponíveis nessa etapa: {sorted(ideb_anos_iniciais['ano'].unique())}")

    print("\n% preenchido de 'taxa_aprovacao' nos anos iniciais:",
          f"{ideb_anos_iniciais['taxa_aprovacao'].notna().mean() * 100:.2f}%")
    print("% preenchido de 'ideb' nos anos iniciais:",
          f"{ideb_anos_iniciais['ideb'].notna().mean() * 100:.2f}%")

    print("\nDistribuição de 'taxa_aprovacao' nos anos iniciais:")
    print(ideb_anos_iniciais["taxa_aprovacao"].describe())

    print("\nDistribuição de 'ideb' nos anos iniciais:")
    print(ideb_anos_iniciais["ideb"].describe())
else:
    print("Nenhum valor de 'anos_escolares' bateu com 'inicia' - preciso olhar os valores reais impressos na Seção 2 e ajustar o filtro manualmente.")

## 5. Checando o JOIN com `escola_completo` via `id_escola` — não vou assumir, vou conferir

Mesma disciplina que usei quando descobri que `alunos` e `escola` não batiam: comparo amostra de valores brutos, distribuição de prefixo de 2 dígitos (deveria bater com código de UF do IBGE dos dois lados, já que ambos são dados oficiais do INEP/Censo Escolar), e a interseção real, contando linhas de verdade.

In [ ]:
CACHE_ESCOLA_COMPLETO = Path("../..") / "data" / "processed" / "escola_completo.parquet"

if CACHE_ESCOLA_COMPLETO.exists():
    print(f"Lendo escola_completo do cache local: {CACHE_ESCOLA_COMPLETO}")
    escola_completo = pd.read_parquet(CACHE_ESCOLA_COMPLETO)
else:
    PREFIXO_BRONZE_ESCOLA_COMPLETO = "bronze/br_inep_censo_escolar/escola_completo/"
    print(f"Cache não encontrado, lendo do S3: s3://{BUCKET}/{PREFIXO_BRONZE_ESCOLA_COMPLETO}")
    escola_completo = _ler_parquet_do_prefixo(BUCKET, PREFIXO_BRONZE_ESCOLA_COMPLETO)
    CACHE_ESCOLA_COMPLETO.parent.mkdir(parents=True, exist_ok=True)
    escola_completo.to_parquet(CACHE_ESCOLA_COMPLETO, index=False)

ideb_escola["id_escola"] = ideb_escola["id_escola"].astype(str)
escola_completo["id_escola"] = escola_completo["id_escola"].astype(str)

print("\nAmostra de id_escola em 'ideb_escola':", sorted(ideb_escola["id_escola"].unique())[:10])
print("Amostra de id_escola em 'escola_completo':", sorted(escola_completo["id_escola"].unique())[:10])

print("\nTamanhos (nº de caracteres) mais comuns em 'ideb_escola':")
print(ideb_escola["id_escola"].str.len().value_counts().head())
print("\nTamanhos (nº de caracteres) mais comuns em 'escola_completo':")
print(escola_completo["id_escola"].str.len().value_counts().head())

print("\nPrefixos de 2 dígitos em 'ideb_escola' (deveriam ser códigos de UF válidos, 11 a 53):")
print(ideb_escola["id_escola"].str[:2].value_counts().sort_index())

In [ ]:
ids_ideb = set(ideb_escola["id_escola"].unique())
ids_escola_completo = set(escola_completo["id_escola"].unique())

interseccao = ids_ideb & ids_escola_completo

print(f"id_escola distintos em 'ideb_escola': {len(ids_ideb):,}")
print(f"id_escola distintos em 'escola_completo': {len(ids_escola_completo):,}")
print(f"Interseção: {len(interseccao):,}")
print(f"% de 'ideb_escola' que encontra correspondência em 'escola_completo': {len(interseccao) / len(ids_ideb) * 100:.2f}%")
print(f"% de 'escola_completo' que encontra correspondência em 'ideb_escola': {len(interseccao) / len(ids_escola_completo) * 100:.2f}%")

## 7. Filtrar só rede pública melhora o preenchimento?

Antes de decidir se uso essa base como alvo, quero ver se o problema de preenchimento que já apareceu (principalmente na rede com poucas escolas) melhora quando eu olho só para a rede pública (municipal + estadual + federal), excluindo a privada — que é uma fatia bem pequena da base e, pela amostra de 200 linhas que já olhei, aparentava ter presença fraca de resultado.

Aqui eu calculo, na base inteira (não só na amostra de 200), o percentual de preenchimento de `taxa_aprovacao` e `ideb` por valor de `rede`, e depois comparo "pública" (tudo que não é privada) contra "privada" — tanto na tabela toda quanto no recorte de anos iniciais.

In [ ]:
def resumo_preenchimento_por_rede(df, nome):
    linhas = []
    for rede_valor, grupo in df.groupby("rede", dropna=False):
        linhas.append({
            "base": nome,
            "rede": rede_valor,
            "linhas": len(grupo),
            "pct_taxa_aprovacao": round(grupo["taxa_aprovacao"].notna().mean() * 100, 2),
            "pct_ideb": round(grupo["ideb"].notna().mean() * 100, 2),
        })
    return pd.DataFrame(linhas)

print("=== Preenchimento por rede — tabela toda ===")
resumo_por_rede_geral = resumo_preenchimento_por_rede(ideb_escola, "geral")
print(resumo_por_rede_geral)

if valor_anos_iniciais:
    print("\n=== Preenchimento por rede — só anos iniciais ===")
    resumo_por_rede_iniciais = resumo_preenchimento_por_rede(ideb_anos_iniciais, "anos_iniciais")
    print(resumo_por_rede_iniciais)

# Agora a comparação direta que me interessa: pública (tudo != privada) vs. privada
def resumo_publica_vs_privada(df, nome):
    eh_privada = df["rede"].astype(str).str.lower() == "privada"
    linhas = []
    for label, mascara in [("pública (municipal+estadual+federal)", ~eh_privada), ("privada", eh_privada)]:
        grupo = df[mascara]
        linhas.append({
            "base": nome,
            "grupo": label,
            "linhas": len(grupo),
            "pct_taxa_aprovacao": round(grupo["taxa_aprovacao"].notna().mean() * 100, 2) if len(grupo) else None,
            "pct_ideb": round(grupo["ideb"].notna().mean() * 100, 2) if len(grupo) else None,
        })
    return pd.DataFrame(linhas)

print("\n=== Pública vs. privada — tabela toda ===")
print(resumo_publica_vs_privada(ideb_escola, "geral"))

if valor_anos_iniciais:
    print("\n=== Pública vs. privada — só anos iniciais ===")
    print(resumo_publica_vs_privada(ideb_anos_iniciais, "anos_iniciais"))

## 8. Decisão confirmada: variável-alvo binária a partir do IDEB

Decidi usar o `ideb` (escala 0 a 10) como base da variável-alvo, binarizada com **6,0** como corte — esse valor não é arbitrário, é a meta nacional de referência do próprio MEC/INEP (aproximadamente a nota dos alunos mais bem colocados do Brasil no PISA).

```
alvo = 1 se ideb >= 6.0
alvo = 0 se ideb < 6.0
```

Essa binarização em si é **pré-processamento** (transformação da variável-resposta para caber no tipo de problema — classificação binária, em vez de regressão sobre uma nota contínua). Ainda não implemento essa coluna aqui — só documento a decisão. A implementação de fato vai para `src/preprocessing` depois que eu terminar as checagens abaixo (grão do ano, filtro de rede, e o filtro pra só escolas que existem na `escola_completo`).

## 9. Rumo à camada silver: filtrar só escolas que existem na `escola_completo`, e olhar de novo todas as variáveis nesse recorte

Já sei da Seção 5 que 95,89% das escolas do `ideb_escola` aparecem na `escola_completo` (a interseção real, calculada com `set` em cima dos `id_escola` de verdade). Antes de decidir que essa vai ser a base da minha camada silver, quero conferir se esse filtro distorce alguma coisa: refaço a EDA (describe/`value_counts`) em cima do subconjunto já filtrado, pra ver se as distribuições continuam parecidas com a tabela cheia da Seção 3/4, ou se a fatia que "sobra" tem algum viés que eu precise documentar.

In [ ]:
ideb_escola_no_censo = ideb_escola[ideb_escola["id_escola"].isin(interseccao)].copy()

print(f"Linhas em ideb_escola_no_censo: {len(ideb_escola_no_censo):,} de {len(ideb_escola):,} "
      f"({len(ideb_escola_no_censo) / len(ideb_escola) * 100:.2f}%)")
print(f"Escolas distintas: {ideb_escola_no_censo['id_escola'].nunique():,} de {ideb_escola['id_escola'].nunique():,}")

colunas_para_descrever = [
    "ano", "sigla_uf", "id_municipio", "rede", "ensino", "anos_escolares",
    "taxa_aprovacao", "indicador_rendimento", "nota_saeb_matematica",
    "nota_saeb_lingua_portuguesa", "nota_saeb_media_padronizada", "ideb", "projecao",
]

for col in colunas_para_descrever:
    print(f"\n--- {col} ---")
    if pd.api.types.is_numeric_dtype(ideb_escola_no_censo[col]):
        print(ideb_escola_no_censo[col].describe())
    else:
        print(ideb_escola_no_censo[col].value_counts(dropna=False).head(20))

## 10. Existe alguma medida de quantidade de alunos por escola?

Olhando o catálogo que recebi do `br_inep_ideb/escola` (Seção 0), não tem nenhuma coluna de quantidade de matrícula/alunos — só `ano`, `sigla_uf`, `id_municipio`, `id_escola`, `rede`, `ensino`, `anos_escolares` e as colunas de resultado. Se eu quiser essa informação, ela só pode vir de um JOIN com a `escola_completo` (que já sei que bate por `id_escola`). Busco aqui, por nome de coluna, candidatas a "quantidade de alunos" dentro da `escola_completo`.

In [ ]:
palavras_chave_quantidade_alunos = ["matricula", "aluno", "estudante", "quantidade_matricula"]

candidatas_quantidade_alunos = sorted({
    c for c in escola_completo.columns
    if any(p in c.lower() for p in palavras_chave_quantidade_alunos)
})

print(f"Colunas candidatas em escola_completo ({len(candidatas_quantidade_alunos)}):")
for c in candidatas_quantidade_alunos:
    print(" -", c)

for c in candidatas_quantidade_alunos:
    print(f"\n--- {c} ---")
    print(f"% preenchido: {escola_completo[c].notna().mean() * 100:.2f}%")
    print(escola_completo[c].describe())

## 11. Qual ano usar? % de preenchimento por ano, já no recorte que interessa (censo ∩ rede pública)

Minha preferência é usar só 2024 (ano mais recente e mais alinhado com o resto do projeto), mas o IDEB é calculado a cada 2 anos em anos ímpares (2005, 2007, ..., 2023...) — se não existir resultado em ano par, ou se o preenchimento em 2024 for ruim, uso o ano mais recente disponível ou o de maior percentual de preenchimento. Calculo aqui o preenchimento por ano já filtrando para escolas que aparecem na `escola_completo` (Seção 9) e só rede pública (mesmo filtro/decisão da Seção 7/12), pra decidir com dado real, não achismo.

In [ ]:
ideb_censo_publica = ideb_escola_no_censo[
    ideb_escola_no_censo["rede"].astype(str).str.lower() != "privada"
].copy()

print(f"Linhas em ideb_censo_publica: {len(ideb_censo_publica):,} de {len(ideb_escola_no_censo):,}")

resumo_por_ano = (
    ideb_censo_publica.groupby("ano")
    .agg(
        linhas=("id_escola", "size"),
        escolas_distintas=("id_escola", "nunique"),
        pct_taxa_aprovacao=("taxa_aprovacao", lambda s: round(s.notna().mean() * 100, 2)),
        pct_ideb=("ideb", lambda s: round(s.notna().mean() * 100, 2)),
    )
    .reset_index()
    .sort_values("ano")
)
print(resumo_por_ano)

print(f"\nAno mais recente disponível: {ideb_censo_publica['ano'].max()}")
print("Ano com maior % de preenchimento de 'ideb':",
      resumo_por_ano.loc[resumo_por_ano["pct_ideb"].idxmax(), "ano"])

**Achado colateral resolvido:** rodei `escola_completo['ano'].value_counts()` (fora deste notebook) e o resultado é limpo — **as 215.545 linhas da `escola_completo` são 100% `ano = 2024`**, sem nenhum outro ano misturado. Isso fecha em definitivo a dúvida que ficou em aberto desde o `04_eda_escola_completo.ipynb` (uma linha de exemplo antiga tinha aparecido com `ano = "2015"`, o que gerou a dúvida — mas era um caso isolado, não representa a base real).

Isso confirma o problema de alinhamento: como `escola_completo` é só 2024 (ano par) e `ideb_escola` só existe em anos ímpares (2005-2025), não existe um "match exato" de ano — preciso escolher o ciclo do IDEB mais próximo pra parear com as features de 2024. Pela tabela acima, **2025 tem o maior preenchimento de `ideb` (71,08%)** dentre os anos mais recentes, e é também o único ciclo *posterior* ao Censo de 2024 — o que teria até uma lógica causal defensável (infraestrutura registrada em 2024 → resultado de um ciclo de avaliação seguinte, em 2025), coerente com uma das perguntas de negócio do projeto ("identificar escolas que podem não atingir metas *futuras*"). A alternativa mais próxima "antes" do Censo seria 2023 (67,44% de preenchimento) — nesse caso as features de 2024 estariam "depois" do resultado que queremos prever, o que é logicamente mais estranho de justificar.

**Decisão final: uso 2025.** Além do preenchimento mais alto e da lógica causal mais defensável, tem um argumento de fundo que reforça essa escolha: as características de infraestrutura de uma escola (água potável, internet, biblioteca, laboratório, esgoto, etc.) não mudam de um ano pro outro com a mesma velocidade que um indicador de desempenho muda — construir uma biblioteca ou levar internet pra uma escola é um investimento que leva tempo e, uma vez feito, tende a se manter por vários anos. Por isso, parear a infraestrutura registrada no Censo de 2024 com o resultado do IDEB de 2025 é uma aproximação razoável: não estou dizendo que a escola "é a mesma" nos dois anos, só que a foto de infraestrutura de 2024 ainda descreve bem a escola de 2025 — o que não seria verdade, por exemplo, se eu estivesse pareando indicadores de desempenho de anos diferentes (esses sim mudam bastante ano a ano). Fecho essa decisão aqui e uso `ANO_ESCOLHIDO = 2025` a partir do notebook 07 em diante.

**Observação nova: `ideb` está sempre preenchido menos que `taxa_aprovacao`.** Isso é esperado pela própria fórmula (Seção 14) — `ideb` só existe quando **as duas** peças existem (`indicador_rendimento` E `nota_saeb_media_padronizada`), enquanto `taxa_aprovacao` só depende do fluxo escolar (mais fácil de estar disponível, já que só precisa do Censo/fluxo, não da aplicação de uma prova). Ou seja, `taxa_aprovacao` preenchido não implica `ideb` preenchido, mas o contrário sim.

Isso levanta uma pergunta legítima: já que `taxa_aprovacao` tem mais dado disponível, dá pra usar ela (ou um corte nela) no lugar do `ideb` como alvo? Tecnicamente sim, mas não decido isso na EDA — é uma decisão de **feature engineering/definição do alvo**, registrada como pendência pra resolver quando eu for de fato montar a coluna de alvo em `src/preprocessing`. Um ponto relevante pra essa decisão futura: ao contrário do IDEB (que tem a meta nacional de referência de 6,0, bem documentada e usada pelo MEC), **não encontrei nenhum corte oficial único de "taxa de aprovação aceitável"** — o MEC divulga médias nacionais de referência (em torno de 90-92% no fundamental, conforme os indicadores mais recentes de rendimento escolar), mas não achei uma meta nacional fixa, tipo o 6,0 do IDEB ou o 743 do SAEB de alfabetização. Se eu decidir usar `taxa_aprovacao` como alvo (ou complementar ao `ideb`) no lugar do IDEB puro, vou precisar definir esse corte com algum outro critério (ex.: mediana/quartil da própria base, ou uma meta estadual/municipal, se existir) — fica registrado como pendência, não decidido agora.

## 12. Sanity check manual — o IDEB dessas escolas bate com a fonte oficial?

Já confirmei que o `id_escola` bate estruturalmente com a `escola_completo` (Seção 5). Mas isso não garante que o *valor* do IDEB está correto — só que o identificador é o mesmo sistema de código. Para uma checagem de sanidade de verdade, pego uma amostra pequena de escolas municipais (rede pública, com `ideb` preenchido) e imprimo `id_escola`, `id_municipio`, `sigla_uf`, `ano` e `ideb`. Com esses códigos em mãos, dá pra conferir manualmente no [QEdu](https://qedu.org.br/) ou na consulta de resultados do INEP se o IDEB mostrado lá bate com o valor que estou vendo aqui na minha base — se não bater, é sinal de que os dados vieram de uma versão diferente/desatualizada.

In [ ]:
amostra_sanity_check = (
    ideb_censo_publica[
        (ideb_censo_publica["rede"].astype(str).str.lower() == "municipal")
        & (ideb_censo_publica["ideb"].notna())
    ]
    .sort_values("ano", ascending=False)
    .drop_duplicates(subset="id_escola")
    .sample(n=5, random_state=42)
    [["id_escola", "id_municipio", "sigla_uf", "ano", "rede", "anos_escolares", "ideb", "taxa_aprovacao"]]
)

print("Escolas para conferência manual (id_escola = código INEP oficial):")
print(amostra_sanity_check.to_string(index=False))

**Sanity check validado.** Consegui confirmar manualmente uma das escolas da amostra (`id_escola = 26034999`): existe de verdade, é a "Escola Municipal Santo Antônio", em Pernambuco — batendo com `sigla_uf = PE` e `rede = municipal` da minha base. Isso confirma que os `id_escola` não são códigos inventados nem de outra fonte — são códigos INEP reais e localizáveis.

## 13. Decisão: excluir a rede privada

Decidi filtrar o modelo só para rede pública (municipal + estadual + federal), excluindo a privada. Dois motivos, um de dado e um conceitual:

- **Dado (Seção 7):** a rede privada tem preenchimento muito pior de `taxa_aprovacao`/`ideb` (19,90%/13,67%, contra 70,32%/61,64% da rede pública) e representa uma fatia mínima da base (6.241 de ~1,3 milhão de linhas, <0,5%). No recorte de anos iniciais ela nem aparece. Ou seja, incluir a privada praticamente não acrescenta dado e ainda piora a qualidade geral.
- **Conceitual:** o objetivo do projeto é apoiar política pública de alfabetização. A rede privada tem outros meios (recursos próprios, mercado) de identificar e corrigir problemas de aprendizagem — o valor de um modelo preditivo é maior justamente na rede pública, onde a gestão tem menos recursos de monitoramento próprio. Faz sentido focar o escopo ali.

## 14. O IDEB é só nota de prova, ou entra infraestrutura? (checagem de vazamento de dado)

Minha preocupação: se o `ideb` já incorporar informação de infraestrutura da escola (internet, biblioteca, laboratório, etc. — que são justamente colunas que tenho na `escola_completo` e pretendo usar como *features*), eu estaria com vazamento de dado (data leakage) — o modelo "acertaria" o alvo usando uma variável que já é, em parte, derivada dele mesmo.

Pesquisei a documentação oficial do INEP sobre o IDEB e a fórmula é bem específica: **IDEB = indicador de rendimento (fluxo escolar/taxa de aprovação) × nota SAEB padronizada de desempenho**. A nota técnica oficial do INEP é explícita que o índice **não** usa infraestrutura da escola, quantidade de matrícula, formação de professores ou qualquer outra característica institucional — só essas duas peças (fluxo e desempenho na prova). Confiro isso na prática abaixo, recalculando o `ideb` a partir de `indicador_rendimento × nota_saeb_media_padronizada` e comparando com o valor que já vem na tabela — se a diferença for essencialmente zero, confirma que não tem nada "escondido" na fórmula além dessas duas colunas.

**Conclusão prática para o meu pipeline:** não tem risco de vazamento vindo da `escola_completo` (infraestrutura) para dentro do `ideb` — são fontes independentes. O vazamento que EU preciso evitar é outro: **não posso usar `taxa_aprovacao`, `indicador_rendimento`, as notas do SAEB nem `ideb`/`projecao` como *features* do modelo**, porque essas colunas compõem o próprio alvo (usar qualquer uma delas como entrada seria o modelo "prever" o alvo usando (quase) o próprio alvo). As features de infraestrutura/recursos da `escola_completo` continuam liberadas, porque são de fato uma fonte de dado independente do IDEB.

In [ ]:
verificacao_formula = ideb_escola_no_censo.dropna(
    subset=["ideb", "indicador_rendimento", "nota_saeb_media_padronizada"]
).copy()

verificacao_formula["ideb_recalculado"] = (
    verificacao_formula["indicador_rendimento"] * verificacao_formula["nota_saeb_media_padronizada"]
)
verificacao_formula["diferenca_absoluta"] = (
    verificacao_formula["ideb"] - verificacao_formula["ideb_recalculado"]
).abs()

print(f"Linhas verificadas: {len(verificacao_formula):,}")
print("\nDiferença absoluta entre 'ideb' informado e 'indicador_rendimento x nota_saeb_media_padronizada':")
print(verificacao_formula["diferenca_absoluta"].describe())
print(f"\n% de linhas com diferença > 0.01: "
      f"{(verificacao_formula['diferenca_absoluta'] > 0.01).mean() * 100:.2f}%")

**Resultado real:** em 787.703 linhas verificadas, a diferença absoluta entre o `ideb` que já vem na tabela e `indicador_rendimento × nota_saeb_media_padronizada` (recalculado por mim) tem média 0,025, mediana 0,025 e máximo 0,05 — numa escala de 0 a 10, isso é no máximo 0,5% da escala, ou seja, praticamente zero. 80,06% das linhas têm diferença acima de 0,01, mas isso **não** indica um terceiro fator escondido na fórmula — é o efeito esperado de arredondamento em cascata: o INEP arredonda `indicador_rendimento` e `nota_saeb_media_padronizada` a um número de casas decimais antes de multiplicar, e arredonda o `ideb` final de novo depois — então recalcular a fórmula "crua" (sem reproduzir esses arredondamentos intermediários exatos) sempre deixa uma diferença residual pequena e sistemática (por isso o máximo é só 0,05, nunca mais que isso). Se houvesse mais alguma variável entrando na conta (ex.: infraestrutura), eu esperaria diferenças bem maiores e mais aleatórias, não esse padrão pequeno e limitado. **Conclusão confirmada:** a fórmula é essencialmente só essas duas peças (fluxo × desempenho), sem infraestrutura ou qualquer outra característica da escola.

## 15. Conclusão

- **Grão real (Seção 2):** `(id_escola, ano, rede, ensino, anos_escolares)` — 1.300.821 linhas, 86.098 `id_escola` distintos, 11 ciclos bienais (2005-2025). Até 6 linhas por `(id_escola, ano)` porque `anos_escolares` separa etapas — esperado, não duplicidade.
- **Preenchimento, tabela toda (Seção 3):** `taxa_aprovacao`/`indicador_rendimento` 70,08%; `ideb`/notas SAEB 61,41-61,42%. `ideb` média 4,67 (escala 0-10).
- **Recorte de anos iniciais (Seção 4):** 691.760 linhas, 66.138 escolas. Preenchimento não piora (66,51% `taxa_aprovacao`, 61,66% `ideb`); `ideb` média 5,06.
- **Interseção com `escola_completo` (Seção 5):** 95,89% das escolas de `ideb_escola` batem com `escola_completo`; 38,30% no sentido inverso (esperado — nem toda escola do Censo tem SAEB aplicado).
- **Pública vs. privada (Seção 7):** pública 70,32%/61,64% de preenchimento (`taxa_aprovacao`/`ideb`); privada só 19,90%/13,67%, e representa <0,5% da base — no recorte de anos iniciais a privada nem aparece.
- **Alvo binário definido (Seção 8):** `ideb >= 6.0` — decisão já tomada.
- **EDA no recorte "só escolas do censo" (Seção 9):** distribuições praticamente idênticas à tabela cheia (`ideb` média 4,68 vs. 4,67) — o filtro não distorce nada.
- **Quantidade de alunos (Seção 10):** não existe em `ideb_escola`, mas existem 91 colunas candidatas em `escola_completo` (`quantidade_matricula_*`) — a decisão de qual/quais usar como "tamanho da escola" fica para a próxima EDA (notebook 07) e o pré-processamento.
- **Ano escolhido (Seção 11):** `escola_completo` é 100% `ano = 2024` (confirmado). Como `ideb_escola` só existe em anos ímpares, decidi usar **2025** — maior preenchimento entre os ciclos recentes (71,08% de `ideb`), posterior ao Censo 2024 (lógica causal mais defensável), e reforçado pelo fato de que infraestrutura escolar muda pouco de um ano pro outro, então parear o Censo de 2024 com o IDEB de 2025 é uma aproximação razoável. Decisão final, usada a partir do notebook 07.
- **Sanity check (Seção 12):** validado manualmente — escola real, localizável, dados batendo.
- **Exclusão da rede privada (Seção 13):** decisão tomada e documentada (dado ruim + racional de foco em política pública).
- **Checagem de leakage do IDEB (Seção 14):** confirmado que `ideb = indicador_rendimento × nota_saeb_media_padronizada`, sem infraestrutura — as diferenças residuais (média 0,025) são só arredondamento em cascata do INEP, não um fator escondido. `taxa_aprovacao`, `indicador_rendimento`, notas SAEB, `ideb` e `projecao` ficam de fora das features (compõem o alvo).
- **`ideb` vs. `taxa_aprovacao` como base do alvo:** `ideb` sempre tem preenchimento menor porque depende de duas peças (fluxo + SAEB), enquanto `taxa_aprovacao` só depende do fluxo. Não existe um corte oficial único de "taxa de aprovação aceitável" (diferente do 6,0 do IDEB) — decisão registrada como pendência de feature engineering, não resolvida aqui.

**Próximo passo, fora deste notebook:** uma nova EDA (notebook 07) explorando as 455 colunas da `escola_completo` por grupo temático (infraestrutura, corpo docente, matrícula, etc.), com correlação contra `ideb` e `taxa_aprovacao`, pra decidir quais features entram no modelo. Só depois disso vira código de pré-processamento em `src/preprocessing` (filtro de rede pública, filtro de escolas do censo, ano escolhido, alvo binário, features selecionadas).